# Lab 0B: Compare Three Models on PII Extraction

Use this notebook after you finish `../lab0_1_environment_setup/03_environment_check.ipynb`.

In this warm-up notebook, you will use three class models from the same Ollama endpoint and compare how they extract PII and device identifiers from a short synthetic case note.

This task is a good fit for Lab 0 because it is:
- easy to run
- relevant to digital forensics
- simple enough to compare by observation before you revise a prompt
- based on synthetic data rather than real personal data

In [ ]:
import json
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from openai import OpenAI

# Purpose of this block: find the repo root so the notebook can load `.env`.
# In this teaching notebook, we assume the current folder is this lab folder,
# so the repo root is one level up.
cwd = Path.cwd().resolve()
repo_root = cwd.parent

if not (repo_root / '.env.example').exists() or not (repo_root / 'src').exists():
    raise FileNotFoundError(
        'Open this notebook from its lab folder so the repo root is one level up.'
    )

config = dotenv_values(repo_root / '.env')
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Default model from .env:', default_model)
print('Ollama endpoint:', ollama_base_url)

## Step 1: Discover Available Models

Run the next cell to list the models available from the configured Ollama endpoint.

In [ ]:
# `.env` uses the OpenAI-compatible `/v1` endpoint for Ollama.
# Replace that ending with `/api/tags` to get the model list.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

response = requests.get(tags_url, timeout=10)
response.raise_for_status()

# Keep both the model names and a simple size lookup for later display.
models_data = response.json().get('models', [])
available_models = []
model_size_lookup = {}

print('Available models:')
for item in models_data:
    model_name = item.get('name')
    if not model_name:
        continue

    available_models.append(model_name)
    size_bytes = item.get('size')
    if isinstance(size_bytes, (int, float)):
        size_text = f'{size_bytes / (1024 ** 3):.2f} GB'
    else:
        size_text = 'size unavailable'

    model_size_lookup[model_name] = size_text
    print(f'{len(available_models)}. {model_name} ({size_text})')

if len(available_models) < 3:
    raise ValueError('This homework needs at least 3 available models.')

## Step 2: Choose Three Models

Use these three models for the comparison so everyone in the class works with the same set:
- `qwen3.5:0.8b`
- `qwen3.5:27b`
- `gemma4:e4b`

The next cell checks whether all three are available at the configured endpoint.

In [ ]:
# Use the fixed class model set instead of auto-picking from the endpoint.
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']

# Report any required models that are missing from the current endpoint.
missing_models = [name for name in models_to_compare if name not in available_models]

print('Models selected for comparison:')
for model_name in models_to_compare:
    size_text = model_size_lookup.get(model_name, 'size unavailable')
    print(f'- {model_name} ({size_text})')

if missing_models:
    raise ValueError(f'These required models are missing from the endpoint: {missing_models}')

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')

## Step 3: Synthetic Case Note

Use the same synthetic case note for all three models.

In [ ]:
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

## Step 4: Shared Extraction Prompt

All three models should answer the **same prompt** so you can compare them fairly.

In [ ]:
comparison_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(comparison_prompt)

In [ ]:
# Function: clean_json_text
# What it does: Turns a raw model reply into a cleaner JSON-looking string.
# Why this notebook uses it: Some models wrap JSON in Markdown fences or add extra text before or after the JSON.
# Parameter: `text` is the raw response string returned by a model.
# Returns: A cleaned string that is easier to pass to `json.loads(...)`.
# Example call: `clean_json_text('```json\n{"names": ["Maya Chen"]}\n```')`
# Example result: `'{"names": ["Maya Chen"]}'`
# Clean a model response so we have a better chance of parsing it as JSON.
def clean_json_text(text: str) -> str:
    # Remove leading and trailing whitespace first.
    cleaned = text.strip()
    # Some models wrap JSON in Markdown code fences such as ```json ... ```.
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    # Keep only the outermost JSON object if extra text appears before or after it.
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

# Function: ask_model
# What it does: Sends one prompt to one model and records what came back.
# Why this notebook uses it: We want each model run to be captured in the same format for fair comparison.
# Parameters: `model_name` is the model to run, and `prompt` is the shared prompt string.
# Returns: A dictionary with the model name, elapsed time, raw text, parsed JSON, and parse error.
# Example call: `ask_model('qwen3.5:0.8b', 'Return valid JSON only.')`
# Example result: `{'model': 'qwen3.5:0.8b', 'seconds': 1.23, 'raw_text': '{...}', 'parsed': {...}, 'parse_error': None}`
# Ask one model to answer the shared prompt and capture both timing and parsing details.
def ask_model(model_name: str, prompt: str) -> dict:
    # Start a timer so we can compare response speed across models.
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    )
    elapsed = perf_counter() - start
    # Save the raw model text before we try to clean or parse it.
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    # Try to turn the cleaned text into a Python dictionary.
    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    # Return one record containing timing, raw text, parsed JSON, and any parse error.
    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

results = []
for model_name in models_to_compare:
    print(f'Running {model_name}...')
    results.append(ask_model(model_name, comparison_prompt))

print('Comparison complete.')

In [ ]:
for result in results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

## Step 5: Build a Simple Comparison Summary

Instead of a detailed grading rubric, this step creates a small summary that is easier to read in a warm-up lab.

For each model, the summary shows:
- response time in seconds
- whether the response could be parsed as JSON
- which keys were returned
- what the model put in `device_ids`

In [ ]:
comparison_summary = []

for result in results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []

    comparison_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

print(json.dumps(comparison_summary, indent=2))

## Reflection Questions

Replace this text with short answers to the questions below.

Use the summary above and the raw outputs to support your answers.

1. Which model was the fastest?
2. Which models returned valid JSON?
3. Did the three models use the same `device_ids` format? Describe one difference you noticed.
4. Which response was easiest to read and why?
5. What is one prompt rule you might add before doing `03_prompt_revision_assignment.ipynb`?

## Submission

Save the notebook with:
- the discovered model list
- the three selected models
- the raw outputs from all three models
- the simple comparison summary
- your short reflection